In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tensorflow.keras.datasets import imdb

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 200

(x_train,y_train),(x_test,y_test)=imdb.load_data(num_words=VOCAB_SIZE)

def pad(x):
    return np.array([i[:MAX_LEN]+[0]*(MAX_LEN-len(i)) for i in x])

x_train=pad(x_train)
x_test=pad(x_test)

x_train=torch.tensor(x_train)
y_train=torch.tensor(y_train)

x_test=torch.tensor(x_test)
y_test=torch.tensor(y_test)

print(x_train.shape)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
torch.Size([25000, 200])


In [ ]:
class CNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb=nn.Embedding(VOCAB_SIZE,128)
        self.conv=nn.Conv1d(128,128,5)
        self.pool=nn.MaxPool1d(2)
        self.fc=nn.Linear(128*98,1)

    def forward(self,x):
        x=self.emb(x)
        x=x.permute(0,2,1)
        x=self.pool(torch.relu(self.conv(x)))
        x=x.view(x.size(0),-1)
        return torch.sigmoid(self.fc(x))

In [ ]:
model=CNN1D().to(device)
loss_fn=nn.BCELoss()
opt=optim.Adam(model.parameters(),0.001)

for e in range(3):
    perm=torch.randperm(len(x_train))
    correct=0

    for i in range(0,4000,64):   # small subset for speed
        idx=perm[i:i+64]
        xb=x_train[idx].to(device)
        yb=y_train[idx].float().unsqueeze(1).to(device)

        opt.zero_grad()
        out=model(xb)
        loss=loss_fn(out,yb)
        loss.backward()
        opt.step()

        correct+=(out>0.5).int().eq(yb.int()).sum().item()

    print("Epoch",e+1,"Acc:",correct/4000)

Epoch 1 Acc: 0.51125
Epoch 2 Acc: 0.64475
Epoch 3 Acc: 0.7185


In [ ]:
with torch.no_grad():
    out=model(x_test[:1000].to(device))
    acc=(out.cpu()>0.5).int().squeeze().eq(y_test[:1000]).sum()/1000
    print("Test Accuracy:",acc.item())

Test Accuracy: 0.7080000042915344


In [ ]:
def predict(review):
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    w=imdb.get_word_index()
    seq=[w.get(i,0) for i in review.lower().split()]
    seq=pad_sequences([seq],MAX_LEN)
    t=torch.tensor(seq).to(device)
    print("Positive" if model(t).item()>0.5 else "Negative")

predict("this movie was fantastic")
predict("worst film ever")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Negative
Negative
